# Linear Algebra Capstone — Content-Based Movie Recommendations

## Objective

Build a small, reproducible recommender from item-feature vectors and a user's liked items; evaluate its rankings on held-out relevance labels.

**Prerequisites:** Linear algebra Notebooks 01–03: shapes, dot products, matrix multiplication, norms, cosine similarity, broadcasting, and distances.

**Working rule:** Complete the starter cells yourself. Verification cells are deliberately placed after each task and will fail until that task is solved. Keep the assertions; catch only failures that the task explicitly asks you to provoke. Use NumPy plus Python's standard library only. After finishing, restart the kernel and run all cells in order.

**Why this matters:** A content-based recommender is an ML workflow: represent items, derive a user profile, score candidates, rank them, and evaluate recommendations. The provided labels are synthetic and belong only in evaluation.


## Problem and dataset

A small movie platform wants recommendations for a user who liked two titles. Each row is one movie; columns are five nonnegative content features: science fiction, mystery, action, family, and comedy. Feature values are editorial tags, not learned embeddings. Use the held-out `relevant_indices` **only for evaluation**, never to construct the user profile or tune feature weights. The catalog is synthetic so it can be committed safely and rerun without downloads.


In [ ]:
import numpy as np

titles = np.array(['Orbit', 'Harbor', 'Signal', 'Garden', 'Nebula',
                   'Laughter', 'Quest', 'Cipher', 'Home', 'Pulse'])
item_features = np.array([
    [1., 0., 0., 1., 0.], [0., 0., 0., 1., 1.],
    [1., 0., 1., 0., 0.], [0., 1., 0., 0., 1.],
    [1., 0., 1., 1., 0.], [0., 0., 0., 0., 1.],
    [0., 1., 1., 0., 0.], [1., 0., 0., 1., 1.],
    [0., 0., 0., 1., 0.], [1., 0., 1., 0., 1.],
])
feature_names = ('sci_fi', 'mystery', 'action', 'family', 'comedy')
liked_indices = np.array([0, 2])
relevant_indices = {4, 7, 9}  # held-out evaluation labels
popularity = np.array([9, 5, 8, 4, 7, 3, 6, 5, 7, 4])
assert item_features.shape == (len(titles), len(feature_names)) == (10, 5)


## Task 1 — Audit vectors and shapes

### Goal

Define the representation and reject malformed catalogs.

### Required work

1. Implement `validate_catalog(matrix, names)` for a nonempty numerical 2D matrix with one name per row and nonnegative features.
2. Write what rows, columns, and a zero vector mean in this dataset.
3. Test a bad name count and a negative feature value.

### Expected output

The provided 10-by-5 catalog is accepted; bad catalogs raise `ValueError`.

### Acceptance criteria

- [ ] No assumption is made that rows are labels.
- [ ] Invalid structure is rejected before scoring.
- [ ] The feature meaning is explained.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def validate_catalog(matrix, names):
    # TODO: return a validated numerical 2D array.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
catalog = validate_catalog(item_features, titles)
assert catalog.shape == (10, 5)
for bad_matrix, bad_names in ((item_features, titles[:-1]),
                              (-item_features, titles)):
    try:
        validate_catalog(bad_matrix, bad_names)
    except ValueError:
        pass
    else:
        raise AssertionError('Malformed catalog was accepted')


## Task 2 — Build a user taste profile

### Goal

Combine liked-item vectors without using evaluation labels.

### Required work

1. Implement `mean_profile(matrix, liked)` as the mean feature vector for valid liked indices.
2. Verify the output shape and calculate one coordinate by hand.
3. Reject empty, duplicate, or out-of-range liked indices.

### Expected output

One 5-element taste vector; sci-fi component is `1.0`.

### Acceptance criteria

- [ ] The profile depends only on liked items.
- [ ] Invalid indices raise `ValueError`.
- [ ] The explanation discusses the simplicity of averaging.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def mean_profile(matrix, liked):
    # TODO: validate liked indices; return a 1D mean profile.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
profile = mean_profile(catalog, liked_indices)
assert profile.shape == (5,)
assert np.isclose(profile[0], 1.)
assert np.allclose(profile, catalog[liked_indices].mean(axis=0))
for bad in ([], [0, 0], [99]):
    try:
        mean_profile(catalog, bad)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid liked indices were accepted')


## Task 3 — Cosine-score the catalog

### Goal

Use dot products and norms to compare item directions with the profile.

### Required work

1. Implement `cosine_scores(matrix, profile)` returning one similarity per row.
2. Set the score of a zero-feature item to `0` and reject a zero profile or feature-count mismatch.
3. Verify a hand-selected item against a direct dot-product calculation.

### Expected output

A 10-element finite score vector in `[-1,1]`.

### Acceptance criteria

- [ ] Result shape is `(10,)`.
- [ ] Zero-item policy is tested.
- [ ] At least one score agrees with a direct calculation.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def cosine_scores(matrix, profile):
    # TODO: validate shapes; use dot products and L2 norms.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
scores = cosine_scores(catalog, profile)
assert scores.shape == (10,) and np.all(np.isfinite(scores))
assert np.all((scores >= -1 - 1e-12) & (scores <= 1 + 1e-12))
direct = np.dot(catalog[4], profile) / (np.linalg.norm(catalog[4]) * np.linalg.norm(profile))
assert np.isclose(scores[4], direct)
assert cosine_scores(np.vstack((catalog, np.zeros(5))), profile)[-1] == 0
for bad_profile in (np.zeros(5), np.ones(4)):
    try:
        cosine_scores(catalog, bad_profile)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid profile was accepted')


## Task 4 — Rank unseen movies

### Goal

Separate scoring from candidate filtering and tie-breaking.

### Required work

1. Implement `top_k_unseen(scores, seen_indices, k)` returning indices ordered by descending score, then ascending index on ties.
2. Exclude liked titles and reject invalid `k`.
3. Report titles and scores of the top three without inspecting relevance labels.

### Expected output

Three unseen recommendation indices in deterministic order.

### Acceptance criteria

- [ ] No liked item is recommended.
- [ ] Tie order is deterministic.
- [ ] Returned indices match a NumPy reference sort.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def top_k_unseen(scores, seen_indices, k):
    # TODO: validate inputs, remove seen items, sort, return 1D integer indices.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
top3 = np.asarray(top_k_unseen(scores, liked_indices, 3))
assert top3.shape == (3,) and not set(top3).intersection(set(liked_indices))
candidates = np.array([i for i in range(len(scores)) if i not in set(liked_indices)])
reference = candidates[np.lexsort((candidates, -scores[candidates]))][:3]
assert np.array_equal(top3, reference)
assert np.array_equal(top_k_unseen([.4, .4, .1], [], 2), [0, 1])


## Task 5 — Evaluate on held-out relevance

### Goal

Use top-k metrics without letting labels leak into scoring.

### Required work

1. Implement `precision_recall_at_k(recommended, relevant)`.
2. Check a known ranking, then evaluate the top-three cosine recommendations.
3. Construct a popularity-only top-three baseline over unseen items and compare both metrics; explain what a single user's labels cannot establish.

### Expected output

Two precision/recall pairs, each in `[0,1]`; known example gives `2/3` for both.

### Acceptance criteria

- [ ] Evaluation labels are used only here.
- [ ] Metrics handle a nonempty relevant set.
- [ ] Baseline and recommender use the same candidate pool.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def precision_recall_at_k(recommended, relevant):
    # TODO: validate inputs; return (precision, recall).
    raise NotImplementedError

# TODO: produce baseline_top3 from popularity over unseen items.
baseline_top3 = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert np.allclose(precision_recall_at_k([4, 1, 7], relevant_indices), (2/3, 2/3))
model_metrics = precision_recall_at_k(top3, relevant_indices)
assert len(model_metrics) == 2 and all(0 <= z <= 1 for z in model_metrics)
assert baseline_top3 is not None and len(baseline_top3) == 3
assert not set(baseline_top3).intersection(set(liked_indices))
baseline_metrics = precision_recall_at_k(baseline_top3, relevant_indices)
assert len(baseline_metrics) == 2 and all(0 <= z <= 1 for z in baseline_metrics)


## Task 6 — Diagnose scale and cold-start behavior

### Goal

Test two limitations of the content representation.

### Required work

1. Multiply one feature column by 10 and recompute profile and cosine rankings; record whether rankings change.
2. Use a user with no liked titles and demonstrate the chosen policy: `mean_profile` should raise `ValueError`, then use popularity as a documented fallback.
3. Explain how feature weighting, sparse tags, and a single-user evaluation limit conclusions.

### Expected output

Two rankings and a cold-start fallback of three unseen titles.

### Acceptance criteria

- [ ] Reweighted scores are recomputed from a new profile.
- [ ] Cold-start error is caught.
- [ ] A fallback is shown without using held-out relevance labels.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: compute weighted_catalog, weighted_profile, weighted_top3, cold_start_top3.
weighted_catalog = weighted_profile = weighted_top3 = cold_start_top3 = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert weighted_catalog is not None and weighted_catalog.shape == catalog.shape
assert not np.allclose(weighted_catalog, catalog)
assert weighted_profile is not None and weighted_profile.shape == profile.shape
assert np.allclose(weighted_profile, weighted_catalog[liked_indices].mean(axis=0))
assert weighted_top3 is not None and len(weighted_top3) == 3
assert cold_start_top3 is not None and len(cold_start_top3) == 3
assert np.array_equal(cold_start_top3, np.lexsort((np.arange(len(popularity)), -popularity))[:3])
try:
    mean_profile(catalog, [])
except ValueError:
    pass
else:
    raise AssertionError('Empty profile must trigger the documented fallback')


## Completion check

- [ ] All task cells are implemented and all provided assertions pass.
- [ ] Expected exceptions are caught inside the relevant test cells.
- [ ] **Restart Kernel and Run All** succeeds in notebook order.
- [ ] Each task has a short written explanation in your own words.
- [ ] No real private data, credentials, or large generated files were added.

**Conceptual exit questions**

1. How does a dot product enter cosine similarity?
2. Why should liked items be removed before ranking?
3. How can changing feature units or weights change recommendations?
4. What does precision@3 measure here, and what can it not prove?

**Final deliverable:** A rerunnable notebook with a validated catalog, user profile, cosine scores, deterministic top-three list, held-out precision/recall, a popularity baseline, and a short recommendation analysis.
